### 7.8 scipy模块包

- scipy是基于numpy的科学运算包，包括科学计算中常用的各种函数。
- scipy的包结构如下：

子模块包 | 描述
--------|-------
scipy.constants | 数学和物理常用的常数
scipy.[fftpack](https://docs.scipy.org/doc/scipy/reference/fftpack.html) | 快速傅里叶变换的函数
scipy.[integrate](https://docs.scipy.org/doc/scipy/reference/integrate.html) | 积分和常微分方程求解
scipy.[**interpolate**](https://docs.scipy.org/doc/scipy/reference/interpolate.html) | 插值和平滑的相关函数
scipy.[optimize](https://docs.scipy.org/doc/scipy/reference/optimize.html) | 函数极值和泛函求解
scipy.[io](https://docs.scipy.org/doc/scipy/reference/io.html) | 常用数据文件读取函数
scipy.[**linalg**](https://docs.scipy.org/doc/scipy/reference/linalg.html) | 线性代数的相关函数
scipy.[**ndimage**](https://docs.scipy.org/doc/scipy/reference/ndimage.html) | 图像处理函数
scip.[**signal**](https://docs.scipy.org/doc/scipy/reference/signal.html) | 信号处理函数
scip.[special](https://docs.scipy.org/doc/scipy/reference/special.html) | 特殊函数
scipy.[**stats**](https://docs.scipy.org/doc/scipy/reference/stats.html) | 统计分布和相关函数

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib notebook

#### 7.8.1 信号处理

卷积（convolution）是通过两个函数f和g生成第三个函数的一种数学算子。表征函数f与经过翻转和平移的g的乘积函数所围成的梯形面积。卷积可以看作是“滑动平均”的推广。
<center><img src="static/ch07/convolution_def.svg" style="width: 400px;"/></center>

- 一维卷积

<center><img src="static/ch07/convolution_1D.gif" style="width: 619px;"/></center>

- 二维卷积

<center><img src="static/ch07/same_padding_no_strides.gif" style="width: 395px;"></center>

**一维时序信号滤波**

In [ ]:
t=np.linspace(-4,4,100)
x=np.sin(t) #正弦信号
xn=x + np.random.randn(len(t))*0.1 # 添加白噪声

In [ ]:
fg, ax = plt.subplots(figsize=(6.5, 2.5), layout='constrained')
ax.plot(x, label='True')
ax.plot(xn, label='Obs')
ax.legend()

**窗口函数（kernel）**

In [ ]:
windows=['flat', 'hanning', 'hamming', 'bartlett', 'blackman']
window_size = 11
fg, ax = plt.subplots(figsize=(6.5, 2.5), layout='constrained');
ax.plot(np.blackman(window_size))

In [ ]:
from scipy.signal import convolve
# w = np.blackman(window_size)
w = np.ones(window_size)
xn_smoothed = convolve(xn, w/w.sum(), mode='same')

In [ ]:
fg, ax = plt.subplots(figsize=(6.5, 2.5), layout='constrained')
ax.plot(xn, label='original')
ax.plot(xn_smoothed, label='smoothed')
ax.legend()

**二维图像边缘提取**

In [ ]:
from scipy import datasets
from scipy.signal import convolve2d
ascent = datasets.ascent()
scharr = np.array([[ -3-3j, 0-10j,  +3 -3j],
                   [-10+0j, 0+ 0j, +10 +0j],
                   [ -3+3j, 0+10j,  +3 +3j]])
scharr = np.ones((5, 5))/25.  # 均值滤波
grad = convolve2d(ascent, scharr, boundary='symm', mode='same')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3), layout='constrained')
ax1.imshow(ascent, cmap='gray', label='Original')
ax2.imshow(np.absolute(grad), cmap='gray', label='Gradient')

#### 7.8.2 图像处理

形态学运算可以对图像进行预处理（去噪声、简化形状）、增强物体结构（抽取骨骼、细化、粗化、凸包）、从背景中分割物体、对物体进行量化描述（面积、周长、投影）

- **膨胀运算**：将结构元素在图像上滑动，将结构元素原点所对应值设为结构元素值为1的区域对应图像区域的最大值
- **腐蚀运算**：腐蚀运算是与膨胀运算相反的操作，即将结构元素原点对应值设为结构元素值为1的区域对应图像区域的最小值

膨胀运算 | 腐蚀运算
:-: | :-: 
![alt](static/ch07/image_dilution.png) | ![alt](static/ch07/image_erosion.png)


- **开运算**：对图像使用同一结构元素先腐蚀再膨胀
- **闭运算**：对图像使用同一结构元素先膨胀再腐蚀

开运算 | 闭运算
:-: | :-: 
![alt](static/ch07/image_open.png) | ![alt](static/ch07/image_close.png)


**膨胀和腐蚀实例**

In [ ]:
from scipy.ndimage import binary_erosion, binary_dilation

In [ ]:
a = np.zeros((7,7), dtype=int)
a[2:5, 2:5] = 1

In [ ]:
kargs = {'cmap':'gray', 'ec':'0.5'}
fg, axs = plt.subplots(1, 3, figsize=(6., 2.), layout='constrained')
axs[0].pcolor(a, **kargs)
axs[1].pcolor(binary_erosion(a).astype(a.dtype), **kargs)
axs[2].pcolor(binary_dilation(a).astype(a.dtype), **kargs)
for ax in axs:
    ax.set_axis_off()

**开和闭实例**

In [ ]:
from scipy.ndimage import binary_opening, binary_closing

In [ ]:
a = np.zeros((7,7), dtype=int)
a[2:5, 2:5] = 1

fg, axs = plt.subplots(1, 3, figsize=(6., 2.), layout='constrained')
axs[0].pcolor(a, **kargs) 
axs[1].pcolor(binary_opening(a).astype(a.dtype), **kargs)
axs[2].pcolor(binary_closing(a).astype(a.dtype), **kargs)
for ax in axs:
    ax.set_axis_off()

#### 7.8.3 插值

- 在数值分析领域中，**插值**（英语：Interpolation），是一种通过已知的、离散的数据点，推求新数据点的过程或方法
- 在求解科学和工程的问题时，通常由采样、实验等方法获得许多数据点。根据这些数据，到一个连续的函数（也就是曲线），这个过程叫做**拟合**

- 一维插值实例（风廓线雷达数据插值）

In [ ]:
winds = np.genfromtxt('../data/Z_RADA_WPRD_LC_ROBS.txt', delimiter=' ',
                      skip_header=3, skip_footer=1, missing_values='/////')
winds[:5, :3]

In [ ]:
fg, axs = plt.subplots(1, 2, layout='constrained', figsize=(5, 3), sharey=True)
axs[0].plot(winds[:, 2], winds[:, 0], '.-', label='speed')
axs[1].plot(winds[:, 1], winds[:, 0], '.-', label='direction')

In [ ]:
from scipy.interpolate import interp1d
hgt_i = np.arange(500, 10e3+1, 500)
spd_f = interp1d(winds[:, 0], winds[:, 2], kind='slinear')
dir_f = interp1d(winds[:, 0], winds[:, 1], kind='slinear')

In [ ]:
fg, axs = plt.subplots(1, 2, layout='constrained', figsize=(5, 3), sharey=True)
axs[0].plot(winds[:, 2], winds[:, 0], '.-', label='speed')
axs[0].plot(spd_f(hgt_i), hgt_i, '*', color='red')
axs[1].plot(winds[:, 1], winds[:, 0], '.-', label='direction')
axs[1].plot(dir_f(hgt_i), hgt_i, '*', color='red')

- 二维插值个例（雨量站插值）

In [ ]:
df = pd.read_csv('../data/20200826040000.csv', index_col=0)
df.head()

In [ ]:
fg, ax = plt.subplots(figsize=(4.5, 4.), layout='constrained')
ax.scatter(df.lon, df.lat, 3., c=df.rain.astype('f4'), 
                   edgecolors='None', alpha=0.8)

In [ ]:
from scipy.interpolate import griddata

x_o, y_o = np.meshgrid(np.arange(118.5, 121.6, 0.2), np.arange(34.8, 37.5, 0.2))
r_o = griddata(np.c_[df.lon, df.lat], df.rain, 
               np.c_[np.ravel(x_o), np.ravel(y_o)], method='linear')

In [ ]:
fg, ax = plt.subplots(figsize=(4.5, 4.), layout='constrained')
ax.pcolormesh(x_o, y_o, r_o.reshape(x_o.shape), cmap='Reds', vmin=0.1, vmax=50.)

## 谢谢
## Thank you for your attention!